In [ ]:
import re
import pandas as pd
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# 1. INPUT TANGGAL EVALUASI & RAW TEXT TRADING PLAN
# ============================================================
tanggal_evaluasi = "2026-09-02"  # Tanggal Acuan Evaluasi / Target Eksekusi (H+0)
JUMLAH_HARI_LOOKAHEAD = 10       # Evaluasi rentang H+0 s/d H+10 hari kalender

In [ ]:
# TEMPEL SELURUH TEKS TRADING PLAN DARI PROSES 1 DI SINI
# (Format sudah disesuaikan dengan output Cut Loss Dinamis dari Proses 1)
raw_text = """
=================================================================
 TRADING PLAN: ESSA.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp695
. Cut Loss Dinamis     : Rp648 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp744 (+7%)
. Trailing Lock        : Rp730 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: SMBR.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp192
. Cut Loss Dinamis     : Rp179 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp205 (+7%)
. Trailing Lock        : Rp202 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: SMGR.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp1,730
. Cut Loss Dinamis     : Rp1,618 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp1,851 (+7%)
. Trailing Lock        : Rp1,816 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: SGER.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp655
. Cut Loss Dinamis     : Rp606 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp701 (+7%)
. Trailing Lock        : Rp688 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: TAPG.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp2,330
. Cut Loss Dinamis     : Rp2,214 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp2,493 (+7%)
. Trailing Lock        : Rp2,446 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: KKES.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp90
. Cut Loss Dinamis     : Rp82 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp96 (+7%)
. Trailing Lock        : Rp94 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: BELI.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp326
. Cut Loss Dinamis     : Rp303 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp349 (+7%)
. Trailing Lock        : Rp342 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: IRSX.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp406
. Cut Loss Dinamis     : Rp367 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp434 (+7%)
. Trailing Lock        : Rp426 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: GTRA.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp274
. Cut Loss Dinamis     : Rp253 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp293 (+7%)
. Trailing Lock        : Rp288 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: NEST.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp750
. Cut Loss Dinamis     : Rp722 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp802 (+7%)
. Trailing Lock        : Rp788 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: AHAP.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp104
. Cut Loss Dinamis     : Rp96 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp111 (+7%)
. Trailing Lock        : Rp109 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================

=================================================================
 TRADING PLAN: AIMS.JK
=================================================================
. Tanggal Sinyal       : 2026-09-01 (Selasa)
. Target Eksekusi      : 2026-09-02 (Rabu)
-----------------------------------------------------------------
. Harga Beli Acuan     : Rp505
. Cut Loss Dinamis     : Rp472 (Berbasis ATR 14, Maks -10%)
. Trailing Trigger     : Rp540 (+7%)
. Trailing Lock        : Rp530 (+5%)
. Batas Hold Max       : 5 Hari Bursa
=================================================================


"""

In [ ]:
# ============================================================
# 2. PARSER TRADING PLAN (DIPERBARUI UNTUK FORMAT ATR)
# ============================================================
def parse_plans(text):
    # Normalisasi spasi dan karakter khusus (antisipasi error OCR/PDF)
    cleaned_text = text.replace('\xa0', ' ').replace('：', ':').replace('，', ',').replace('．', '.')
    blocks = cleaned_text.split("TRADING PLAN:")
    plans = []

    for block in blocks[1:]: # Skip block pertama yang biasanya kosong
        ticker = re.search(r"^\s*([A-Za-z0-9\.]+)", block)
        buy = re.search(r"Harga Beli\s*Acuan\s*:?\s*R[pP]\s*([\d,\.]+)", block, re.IGNORECASE)

        # Regex ini sekarang fleksibel membaca "Cut Loss (-3%)" ATAU "Cut Loss Dinamis"
        cut_loss = re.search(r"Cut Loss[^\n:]*:?\s*R[pP]\s*([\d,\.]+)", block, re.IGNORECASE)
        lock = re.search(r"Trailing Lock[^\n:]*:?\s*R[pP]\s*([\d,\.]+)", block, re.IGNORECASE)

        if ticker and buy and cut_loss and lock:
            code_raw = ticker.group(1).strip()
            ticker_formatted = code_raw if code_raw.endswith(".JK") else f"{code_raw}.JK"

            # Fungsi helper untuk membersihkan format angka (ribuan)
            def clean_number(num_str):
                num_str = num_str.replace(',', '')
                # Jika ada titik dan 3 angka di belakangnya, anggap pemisah ribuan (format ID)
                if '.' in num_str and len(num_str.split('.')[-1]) == 3:
                    num_str = num_str.replace('.', '')
                return int(num_str)

            plans.append({
                "ticker": ticker_formatted,
                "code": ticker_formatted.replace(".JK", ""),
                "buy": clean_number(buy.group(1)),
                "cut_loss": clean_number(cut_loss.group(1)),
                "lock": clean_number(lock.group(1))
            })
    return plans

parsed_data = parse_plans(raw_text)

if not parsed_data:
    print("❌ Teks gagal dibaca. Pastikan format teks sesuai dengan output Proses 1.")
else:
    df = pd.DataFrame(parsed_data)
    tickers = df["ticker"].tolist()

    # ============================================================
    # 3. AMBIL DATA HISTORIS (H+0 s/d H+N)
    # ============================================================
    t_start = pd.to_datetime(tanggal_evaluasi)
    t_end = t_start + pd.Timedelta(days=JUMLAH_HARI_LOOKAHEAD)
    start_str = t_start.strftime('%Y-%m-%d')
    end_str = t_end.strftime('%Y-%m-%d')

    print(f"Membaca {len(tickers)} saham... Mengambil data dari {start_str} s/d {end_str}...")

    downloaded_data = yf.download(tickers, start=start_str, end=end_str, group_by='ticker', auto_adjust=False, progress=False)

    highest_high_list = []
    lowest_low_list = []
    tgl_open_list = []
    status_list = []
    tgl_tereksekusi_list = []
    pnl_persen_list = [] # [BARU] Untuk tracking persentase profit/loss

    for idx, row in df.iterrows():
        ticker = row["ticker"]
        try:
            df_tk = downloaded_data if len(tickers) == 1 else downloaded_data[ticker].dropna()

            if isinstance(df_tk.columns, pd.MultiIndex):
                df_tk.columns = df_tk.columns.get_level_values(0)

            if df_tk.empty:
                highest_high_list.append(None)
                lowest_low_list.append(None)
                tgl_open_list.append("-")
                status_list.append("DATA TIDAK ADA")
                tgl_tereksekusi_list.append("-")
                pnl_persen_list.append(0)
                continue

            highest_high_list.append(df_tk["High"].max())
            lowest_low_list.append(df_tk["Low"].min())

            # ============================================================
            # 4. EVALUASI STATUS (PROFIT / LOSS / HOLD)
            # ============================================================
            tgl_open = "-"
            status = "HOLD"
            tgl_kejadian = "-"
            is_opened = False
            pnl_persen = 0.0

            for date, price_row in df_tk.iterrows():
                date_str = date.strftime('%Y-%m-%d')

                # Cek kapan posisi Open pertama kali terjadi (Harga Low <= Harga Beli Acuan)
                if not is_opened and price_row["Low"] <= row["buy"]:
                    tgl_open = date_str
                    is_opened = True

                if is_opened:
                    # Prioritas 1: Menyentuh/Melewati Target Profit (Lock)
                    if price_row["High"] >= row["lock"]:
                        status = "PROFIT"
                        tgl_kejadian = date_str
                        pnl_persen = ((row["lock"] - row["buy"]) / row["buy"]) * 100
                        break
                    # Prioritas 2: Menyentuh/Turun ke Cut Loss Dinamis
                    elif price_row["Low"] <= row["cut_loss"]:
                        status = "LOSS"
                        tgl_kejadian = date_str
                        pnl_persen = ((row["cut_loss"] - row["buy"]) / row["buy"]) * 100
                        break

            tgl_open_list.append(tgl_open)
            status_list.append(status if is_opened else "BELUM OPEN")
            tgl_tereksekusi_list.append(tgl_kejadian)
            pnl_persen_list.append(round(pnl_persen, 2))

        except Exception as e:
            highest_high_list.append(None)
            lowest_low_list.append(None)
            tgl_open_list.append("-")
            status_list.append("ERROR")
            tgl_tereksekusi_list.append("-")
            pnl_persen_list.append(0)

    # Masukkan hasil ke DataFrame
    df["Highest High"] = highest_high_list
    df["Lowest Low"] = lowest_low_list
    df["Tanggal Posisi Open"] = tgl_open_list
    df["Status Evaluasi"] = status_list
    df["Tanggal Tereksekusi"] = tgl_tereksekusi_list
    df["PnL (%)"] = pnl_persen_list

    # ============================================================
    # 5. TAMPILKAN HASIL EVALUASI & RINGKASAN ANALITIK
    # ============================================================
    print("\n" + "="*80)
    print("=== HASIL EVALUASI TRADING PLAN ===")
    print("="*80)

    cols_to_display = [
        "code", "buy", "cut_loss", "lock",
        "Highest High", "Lowest Low",
        "Tanggal Posisi Open", "Status Evaluasi", "Tanggal Tereksekusi"
    ]

    # 1. Ambil subset kolom dan buat salinan agar tidak mengubah DataFrame asli
    df_display = df[cols_to_display].copy()

    # 2. Ubah Index mulai dari 1 (agar lebih natural dibaca manusia)
    df_display.index = df_display.index + 1
    df_display.index.name = 'No'

    # 3. [OPSIONAL] Format kolom harga agar ada pemisah ribuan (lebih mudah dibaca)
    price_cols = ["buy", "cut_loss", "lock", "Highest High", "Lowest Low"]
    for col in price_cols:
        df_display[col] = df_display[col].apply(lambda x: f"{x:,.0f}" if pd.notnull(x) else "-")

    # 4. Fungsi untuk mewarnai kolom Status Evaluasi secara visual
    def highlight_status(val):
        if val == 'PROFIT':
            return 'color: #155724; background-color: #d4edda; font-weight: bold'  # Hijau
        elif val == 'LOSS':
            return 'color: #721c24; background-color: #f8d7da; font-weight: bold'  # Merah
        elif val in ['HOLD', 'BELUM OPEN']:
            return 'color: #856404; background-color: #fff3cd'                     # Kuning
        else:
            return 'color: gray'

    # 5. Terapkan styling dan tampilkan dengan display()
    try:
        from IPython.display import display
        styled_df = df_display.style.map(highlight_status, subset=['Status Evaluasi'])
        display(styled_df)
    except ImportError:
        # Fallback jika dijalankan di terminal biasa (bukan Jupyter/Colab)
        print(df_display.to_string())